# FLUX v2 — Phase 2.5: Wave Recurrent Unit (WRU)

**Goal:** Train a FLUX-native recurrent cell that predicts the **next wave** given a prefix context.

```
text → CSE (frozen) → WaveChunker (frozen) → chunk_waves[:n]
  → mean → WaveToField (frozen) → field_context [512]
  → WRU → predicted_wave [432]
  → WaveToText (frozen) → decoded bytes
```

The WRU uses **physics-native operations** instead of GRU gates:
- **Per-band interference gates** (cosine similarity, not sigmoid)
- **Wave superposition** (constructive/destructive, not linear interpolation)
- **Energy conservation** (thermodynamic bound, not gradient clipping)
- **Hidden state IS a valid wave** (decodable at every step)

**Acceptance criteria:**
- Decode gate avg byte accuracy ≥ 60%, min ≥ 30%
- Different contexts → different predictions (avg pairwise cosine < 0.90)
- WRU output always a valid wave (bounded energy, non-degenerate bands)

**Ordered cells:** Setup → Refresh → Hardware → Smoke Test → Training → Diagnostics
→ Upload → Test 1 → Test 2 → Test 3 → Demo 1 → Demo 2 → Save Results → Final Summary

---
> **Requires Phase 1 + Phase 2 v2 checkpoints.**
> HuggingFace: [UnseenGAP/FLUX](https://huggingface.co/UnseenGAP/FLUX)
> GitHub: [Unseengap/FLUX @ v2](https://github.com/Unseengap/FLUX/tree/v2)

## Cell 1 — SETUP & CLONE
**Run ONCE on a fresh session.** Installs deps, clones repo, adds to `sys.path`.
Do NOT re-run after Cell 2 (Refresh).

In [ ]:
# ── Cell 1: SETUP & CLONE ─────────────────────────────────────────────────────
# Run ONCE on a fresh session. Do NOT re-run after Refresh (Cell 2).
# ─────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

# ── Detect runtime environment ────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    RUNTIME  = 'kaggle'
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    RUNTIME  = 'colab'
    WORK_DIR = '/content'
else:
    RUNTIME  = 'local'
    WORK_DIR = os.path.expanduser('~')

REPO_PATH   = f'{WORK_DIR}/FLUX'
GITHUB_REPO = 'https://github.com/Unseengap/FLUX.git'
BRANCH      = 'v2'

print(f"  Runtime  : {RUNTIME}")
print(f"  WORK_DIR : {WORK_DIR}")
print(f"  REPO_PATH: {REPO_PATH}")

# ── Install dependencies ───────────────────────────────────────────────────────
print("\n  Installing dependencies...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'torchaudio',
    'transformers', 'huggingface_hub',
    'numpy', 'scipy', 'matplotlib',
    'tqdm', 'rich',
    'datasets',
], check=True)
print("  ✓ Dependencies installed")

# ── Clone repo ────────────────────────────────────────────────────────────────
if os.path.exists(REPO_PATH):
    print(f"  ✓ Repo already present at {REPO_PATH} — skipping clone")
else:
    print(f"  Cloning FLUX [{BRANCH}] → {REPO_PATH} ...")
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', GITHUB_REPO, REPO_PATH],
        check=True,
    )
    print(f"  ✓ Repo cloned")

# ── Add repo to sys.path ─────────────────────────────────────────────────────
for _p in [REPO_PATH,
           f'{REPO_PATH}/v2/phase1',
           f'{REPO_PATH}/v2/phase2',
           f'{REPO_PATH}/v2/phase2_5']:
    if _p not in sys.path:
        sys.path.insert(0, _p)
print(f"  ✓ v2/phase1, /phase2, /phase2_5 added to sys.path")

# ── Optionally install as editable package ────────────────────────────────────
setup_py = os.path.join(REPO_PATH, 'setup.py')
if os.path.exists(setup_py):
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_PATH],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print("  ✓ setup.py installed (editable)")
        else:
            print(f"  ⚠ Editable install failed (using sys.path) — {result.stderr.strip()[:120]}")
    except Exception as _e:
        print(f"  ⚠ Editable install skipped: {_e}")

print("\n  ✓ SETUP COMPLETE — proceed to Cell 2 (REFRESH)")

## Cell 2 — REFRESH ⟳
**Run at the start of every session and after every bug fix.**
Clears namespace, pulls latest code, wipes stale results, re-defines all constants.

In [ ]:
# ── Cell 2: REFRESH ───────────────────────────────────────────────────────────
# Run before EVERY training session. Clears stale state and pulls latest code.
# ─────────────────────────────────────────────────────────────────────────────

# 1. Clear Python namespace
%reset -f

# 2. Re-import essentials and re-define all constants
import os, gc, sys, shutil, subprocess
import torch

# ── Runtime detection ─────────────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    RUNTIME  = 'kaggle'
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    RUNTIME  = 'colab'
    WORK_DIR = '/content'
else:
    RUNTIME  = 'local'
    WORK_DIR = os.path.expanduser('~')

# ── Constants ─────────────────────────────────────────────────────────────────
REPO_PATH      = f'{WORK_DIR}/FLUX'
RESULTS_DIR    = f'{REPO_PATH}/v2/V2_results/phase2_5'
CHECKPOINT_DIR = f'{REPO_PATH}/checkpoints'
LOGS_DIR       = f'{RESULTS_DIR}/logs'
GRAPHS_DIR     = f'{RESULTS_DIR}/graphs'

HF_TOKEN       = os.environ.get('HF_TOKEN', '')
GITHUB_TOKEN   = os.environ.get('GITHUB_TOKEN', '')
HF_USER        = 'UnseenGAP'
HF_REPO_ID     = 'UnseenGAP/FLUX'
HF_CKPT_PATH   = 'v2/phase2_5_v2.phase.pt'
HF_P1_PATH     = 'v2/phase1_v2.phase.pt'
HF_P2_PATH     = 'v2/phase2_v2.phase.pt'
LOCAL_CKPT     = f'{CHECKPOINT_DIR}/phase2_5_v2.phase.pt'
LOCAL_P1_CKPT  = f'{CHECKPOINT_DIR}/phase1_v2.phase.pt'
LOCAL_P2_CKPT  = f'{CHECKPOINT_DIR}/phase2_v2.phase.pt'

GITHUB_REPO    = 'https://github.com/Unseengap/FLUX.git'
BRANCH         = 'v2'

PHASE            = 2.5
TOTAL_WAVE_DIM   = 432
FIELD_DIM        = 512

# Add repo to sys.path
for _p in [REPO_PATH,
           f'{REPO_PATH}/v2/phase1',
           f'{REPO_PATH}/v2/phase2',
           f'{REPO_PATH}/v2/phase2_5']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 3. Free GPU VRAM + CPU RAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("  ✓ GPU VRAM cleared, Python GC collected")

# 4. Pull latest code from GitHub
if os.path.exists(REPO_PATH):
    result = subprocess.run(
        ['git', '-C', REPO_PATH, 'pull', 'origin', BRANCH],
        capture_output=True, text=True,
    )
    print(f"  git pull: {result.stdout.strip() or result.stderr.strip()}")
else:
    print("  ⚠ Repo not found — run Cell 1 (SETUP & CLONE) first")

# 5. Wipe previous results and recreate clean directory
if os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
    print(f"  ✓ Cleared {RESULTS_DIR}")
for _d in [RESULTS_DIR, LOGS_DIR, GRAPHS_DIR, CHECKPOINT_DIR]:
    os.makedirs(_d, exist_ok=True)

# 6. Set tokens in environment
os.environ['HF_TOKEN']     = HF_TOKEN
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

# 7. Verify key source files exist
_required_files = [
    f'{REPO_PATH}/v2/phase1/cse.py',
    f'{REPO_PATH}/v2/phase1/wave_chunker.py',
    f'{REPO_PATH}/v2/phase1/wave_to_text.py',
    f'{REPO_PATH}/v2/phase1/decode_gate.py',
    f'{REPO_PATH}/v2/phase1/train_codec.py',
    f'{REPO_PATH}/v2/phase2/wave_to_field.py',
    f'{REPO_PATH}/v2/phase2/field_to_wave.py',
    f'{REPO_PATH}/v2/phase2/train_field.py',
    f'{REPO_PATH}/v2/phase2_5/wave_recurrent_unit.py',
    f'{REPO_PATH}/v2/phase2_5/train_wru.py',
    f'{REPO_PATH}/flux_utils.py',
]
_missing = [f for f in _required_files if not os.path.exists(f)]
if _missing:
    print(f"  ✗ MISSING FILES — run Cell 1 first:")
    for f in _missing:
        print(f"      {f}")
else:
    print(f"  ✓ All {len(_required_files)} source files verified")

# 8. Summary
print(f"""
  ╔══════════════════════════════════════════════╗
  ║   FLUX v2 Phase 2.5 — REFRESH DONE          ║
  ╠══════════════════════════════════════════════╣
  ║  Runtime      : {RUNTIME:<28s}  ║
  ║  REPO_PATH    : {REPO_PATH[:28]:<28s}  ║
  ║  RESULTS      : v2/V2_results/phase2_5       ║
  ║  HF_TOKEN     : {'✓ set' if HF_TOKEN else '✗ missing':<28s}  ║
  ║  GITHUB_TOKEN : {'✓ set' if GITHUB_TOKEN else '✗ missing':<28s}  ║
  ╚══════════════════════════════════════════════╝
""")

## Cell 3 — HARDWARE & CREDENTIALS

In [ ]:
# ── Cell 3: HARDWARE & CREDENTIALS ────────────────────────────────────────────
import sys, os, shutil
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import torch
from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=2)
log.cell_start("Cell 3 — Hardware & Credentials (Phase 2.5)")

# ── Runtime summary ───────────────────────────────────────────────────────────
log.info(f"Runtime   : {RUNTIME}")
log.info(f"WORK_DIR  : {WORK_DIR}")
log.info(f"REPO_PATH : {REPO_PATH}")

# ── GPU / device ──────────────────────────────────────────────────────────────
DEVICE = get_device()
log.info(f"Device    : {DEVICE}")

if torch.cuda.is_available():
    gpu_name   = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram  = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated(0)) / 1e9
    log.success(f"GPU: {gpu_name}")
    log.metric("Total VRAM", f"{total_vram:.1f} GB")
    log.metric("Free VRAM",  f"{free_vram:.1f} GB")
    if total_vram < 8:
        log.warning("< 8 GB VRAM — consider smaller batch")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    log.success("Apple MPS (Metal) available")
else:
    log.warning("No GPU detected — training will be slow on CPU")

# ── Resolve HF token ──────────────────────────────────────────────────────────
_resolved_hf = HF_TOKEN
if not _resolved_hf:
    try:
        from google.colab import userdata
        _resolved_hf = userdata.get('HF_TOKEN') or ''
        if _resolved_hf: log.info("HF_TOKEN loaded from Colab secrets")
    except Exception:
        pass
if not _resolved_hf:
    try:
        from kaggle_secrets import UserSecretsClient
        _resolved_hf = UserSecretsClient().get_secret('HF_TOKEN') or ''
        if _resolved_hf: log.info("HF_TOKEN loaded from Kaggle secrets")
    except Exception:
        pass

if not _resolved_hf:
    log.warning("HF_TOKEN not found — HF upload/download will fail")
else:
    HF_TOKEN = _resolved_hf.strip()
os.environ['HF_TOKEN'] = HF_TOKEN

# ── Resolve GitHub token ──────────────────────────────────────────────────────
_resolved_gh = GITHUB_TOKEN
if not _resolved_gh:
    try:
        from google.colab import userdata
        _resolved_gh = userdata.get('GITHUB_TOKEN') or ''
        if _resolved_gh: log.info("GITHUB_TOKEN loaded from Colab secrets")
    except Exception:
        pass
if not _resolved_gh:
    try:
        from kaggle_secrets import UserSecretsClient
        _resolved_gh = UserSecretsClient().get_secret('GITHUB_TOKEN') or ''
        if _resolved_gh: log.info("GITHUB_TOKEN loaded from Kaggle secrets")
    except Exception:
        pass

if not _resolved_gh:
    log.warning("GITHUB_TOKEN not found — Cell 13 git push will be skipped")
else:
    GITHUB_TOKEN = _resolved_gh.strip()

# ── HuggingFace login ─────────────────────────────────────────────────────────
if HF_TOKEN:
    try:
        import huggingface_hub
        huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
        log.success(f"HuggingFace authenticated as {HF_USER}")
    except Exception as e:
        log.warning(f"HuggingFace login failed: {e}")
else:
    log.warning("Skipping HuggingFace login — no token available")

# ── Download Phase 1 checkpoint ───────────────────────────────────────────────
_p1_canonical = f'{CHECKPOINT_DIR}/phase1_v2.phase.pt'
_p1_mirrored  = f'{CHECKPOINT_DIR}/v2/phase1_v2.phase.pt'

if os.path.exists(_p1_canonical):
    _mb = os.path.getsize(_p1_canonical) / 1e6
    log.success(f"Phase 1 checkpoint present ({_mb:.1f} MB)")
elif os.path.exists(_p1_mirrored):
    shutil.copy2(_p1_mirrored, _p1_canonical)
    log.success(f"Phase 1 copied from mirrored path")
elif HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        log.info("Downloading Phase 1 checkpoint from HuggingFace...")
        _dl = hf_hub_download(
            repo_id=HF_REPO_ID, filename=HF_P1_PATH,
            token=HF_TOKEN, local_dir=CHECKPOINT_DIR,
        )
        _dl = os.path.realpath(_dl)
        if _dl != _p1_canonical:
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)
            shutil.copy2(_dl, _p1_canonical)
        _mb = os.path.getsize(_p1_canonical) / 1e6
        log.success(f"Phase 1 checkpoint downloaded ({_mb:.1f} MB)")
    except Exception as _e:
        log.error(f"Phase 1 download FAILED: {_e}")
else:
    log.warning("Cannot download Phase 1 — no HF_TOKEN")

# ── Download Phase 2 checkpoint ───────────────────────────────────────────────
_p2_canonical = f'{CHECKPOINT_DIR}/phase2_v2.phase.pt'
_p2_mirrored  = f'{CHECKPOINT_DIR}/v2/phase2_v2.phase.pt'

if os.path.exists(_p2_canonical):
    _mb = os.path.getsize(_p2_canonical) / 1e6
    log.success(f"Phase 2 checkpoint present ({_mb:.1f} MB)")
elif os.path.exists(_p2_mirrored):
    shutil.copy2(_p2_mirrored, _p2_canonical)
    log.success(f"Phase 2 copied from mirrored path")
elif HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        log.info("Downloading Phase 2 checkpoint from HuggingFace...")
        _dl = hf_hub_download(
            repo_id=HF_REPO_ID, filename=HF_P2_PATH,
            token=HF_TOKEN, local_dir=CHECKPOINT_DIR,
        )
        _dl = os.path.realpath(_dl)
        if _dl != _p2_canonical:
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)
            shutil.copy2(_dl, _p2_canonical)
        _mb = os.path.getsize(_p2_canonical) / 1e6
        log.success(f"Phase 2 checkpoint downloaded ({_mb:.1f} MB)")
    except Exception as _e:
        log.error(f"Phase 2 download FAILED: {_e}")
else:
    log.warning("Cannot download Phase 2 — no HF_TOKEN")

log.cell_end("Cell 3 — Hardware & Credentials (Phase 2.5)", "PASS")

## Cell 4 — SMOKE TEST
Verifies all Phase 2.5 imports work and a minimal WRU forward pass produces correct shapes.
Also verifies Phase 1 + Phase 2 codecs load (required for training). Must pass before training.

In [ ]:
# ── Cell 4: SMOKE TEST ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

log.cell_start("Cell 4 — Smoke Test (Phase 2.5)")
_all_pass = True

# ── Import checks ─────────────────────────────────────────────────────────────
_imports = [
    # Phase 1
    ('cse',          'ContinuousSemanticEncoder'),
    ('wave_chunker', 'WaveChunker'),
    ('wave_to_text', 'WaveToText'),
    ('decode_gate',  'byte_accuracy'),
    ('train_codec',  'load_phase1_checkpoint'),
    # Phase 2
    ('wave_to_field','WaveToField'),
    ('field_to_wave','FieldToWave'),
    # Phase 2.5
    ('wave_recurrent_unit', 'WaveRecurrentUnit'),
    ('wave_recurrent_unit', 'BAND_SLICES'),
    ('train_wru',           'train_wru'),
    ('train_wru',           'load_phase1_and_phase2'),
    ('train_wru',           'sample_batch'),
]

for module_name, symbol_name in _imports:
    try:
        _mod = __import__(module_name, fromlist=[symbol_name])
        getattr(_mod, symbol_name)
        print(f"  ✓ import {module_name}.{symbol_name}")
    except Exception as e:
        print(f"  ✗ import {module_name}.{symbol_name}  →  {e}")
        _all_pass = False

# ── Forward pass: WRU with random input ───────────────────────────────────────
import torch
from wave_recurrent_unit import WaveRecurrentUnit

try:
    _wru = WaveRecurrentUnit()
    _ctx = torch.randn(4, 512)            # [B=4, field_dim=512]
    _pred, _state = _wru(_ctx)             # [B=4, 432]
    assert _pred.shape == (4, 432), f"Expected (4, 432), got {_pred.shape}"
    assert _state.shape == (4, 432), f"Expected (4, 432), got {_state.shape}"
    _params = _wru.count_parameters()
    print(f"  ✓ WRU forward: input [4, 512] → output {list(_pred.shape)}")
    print(f"  ✓ WRU parameters: {_params:,}")
except Exception as e:
    print(f"  ✗ WRU forward failed: {e}")
    _all_pass = False

# ── Energy constraint check ───────────────────────────────────────────────────
try:
    _big_ctx = torch.randn(2, 512) * 100  # Extreme input
    _pred_big, _ = _wru(_big_ctx)
    _energy = (_pred_big ** 2).sum(dim=-1)
    assert _energy.max().item() <= _wru.energy_cap * 1.2, \
        f"Energy {_energy.max():.1f} exceeds cap {_wru.energy_cap}"
    print(f"  ✓ Energy constraint: max={_energy.max():.1f} (cap={_wru.energy_cap})")
except Exception as e:
    print(f"  ✗ Energy constraint check failed: {e}")
    _all_pass = False

# ── Phase 1 + Phase 2 checkpoint loading ──────────────────────────────────────
try:
    from train_wru import load_phase1_and_phase2
    _components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
    print(f"  ✓ Phase 1 + Phase 2 loaded: {list(_components.keys())}")
except Exception as e:
    print(f"  ✗ Phase 1+2 loading failed: {e}")
    _all_pass = False

# ── Quick chunk_with_bytes sanity check ───────────────────────────────────────
try:
    _cse = _components['cse']
    _chunker = _components['chunker']
    _w2f = _components['w2f']

    _test_text = "The cat sat on the mat"
    _wave = _cse.encode(_test_text)
    _pairs = _chunker.chunk_with_bytes(_wave.full, _test_text.encode('utf-8'))
    assert len(_pairs) >= 2, f"Need ≥2 chunks, got {len(_pairs)}"

    _cw = torch.stack([p[0] for p in _pairs])
    _prefix_ctx = _w2f(_cw[:2].mean(dim=0)).unsqueeze(0)  # [1, 512]
    _pred_test, _ = _wru(_prefix_ctx)
    print(f"  ✓ End-to-end: '{_test_text}' → {len(_pairs)} chunks → prefix → WRU → {list(_pred_test.shape)}")
except Exception as e:
    print(f"  ✗ End-to-end check failed: {e}")
    _all_pass = False

# ── Verdict ───────────────────────────────────────────────────────────────────
if _all_pass:
    log.success("SMOKE TEST PASSED — all imports and shapes OK")
    log.cell_end("Cell 4 — Smoke Test (Phase 2.5)", "PASS")
else:
    log.error("SMOKE TEST FAILED — fix errors before training")
    log.cell_end("Cell 4 — Smoke Test (Phase 2.5)", "FAIL")
    raise RuntimeError("Smoke test failed — do not proceed to training.")

## Cell 5 — TRAINING
Trains the **WaveRecurrentUnit** for single-step next-wave prediction.
Phase 1 (CSE, WaveChunker, WTT) and Phase 2 (WaveToField, FieldToWave) are **FROZEN**.
Default: **30,000 steps** (~30 min on T4/L4). Decode loss active from step 1.

In [ ]:
# ── Cell 5: TRAINING ──────────────────────────────────────────────────────────
import sys, os, importlib
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

# Force-reload training module (avoids stale bytecode from failed prior run)
import train_wru as _tw_module
importlib.reload(_tw_module)
from train_wru import train_wru

import torch
from flux_utils import PhaseLogger

log.cell_start("Cell 5 — Training (Phase 2.5)")

# ── Hyperparameters ───────────────────────────────────────────────────────────
TRAIN_STEPS       = 30_000
BATCH_SIZE        = 64
LEARNING_RATE     = 5e-4
LOG_EVERY         = 500
SAVE_EVERY        = 5_000
GATE_CHECK_EVERY  = 2_500
WAVE_MSE_WEIGHT   = 0.1    # Regularizer only — was drowning decode signal at 1.0
COSINE_WEIGHT     = 1.0    # Direction matters more than magnitude
DECODE_WEIGHT     = 5.0    # TEXT fidelity is the ACTUAL objective

log.info(f"Steps            : {TRAIN_STEPS:,}")
log.info(f"Batch size       : {BATCH_SIZE}")
log.info(f"Learning rate    : {LEARNING_RATE}")
log.info(f"Device           : {DEVICE}")
log.info(f"wave_mse_weight  : {WAVE_MSE_WEIGHT}")
log.info(f"cosine_weight    : {COSINE_WEIGHT}")
log.info(f"decode_weight    : {DECODE_WEIGHT}")
log.info(f"Checkpoint out   : {LOCAL_CKPT}")
log.separator("Starting Phase 2.5 WRU training...")

# ── Run training ──────────────────────────────────────────────────────────────
_trained_wru = train_wru(
    steps            = TRAIN_STEPS,
    batch_size       = BATCH_SIZE,
    device           = DEVICE,
    lr               = LEARNING_RATE,
    log_every        = LOG_EVERY,
    save_every       = SAVE_EVERY,
    gate_check_every = GATE_CHECK_EVERY,
    wave_mse_weight  = WAVE_MSE_WEIGHT,
    cosine_weight    = COSINE_WEIGHT,
    decode_weight    = DECODE_WEIGHT,
    upload_hf        = False,   # Upload handled in Cell 7
    hf_token         = HF_TOKEN,
)

log.success("Training complete")
log.metric("Checkpoint", LOCAL_CKPT)
log.cell_end("Cell 5 — Training (Phase 2.5)", "PASS")

## Cell 6 — TRAINING DIAGNOSTICS
**Run immediately after training.** Checks all show-stoppers before uploading.
Any `FAIL` blocks progression — do not proceed to Cell 7.

In [ ]:
# ── Cell 6: TRAINING DIAGNOSTICS ──────────────────────────────────────────────
import json, math, torch, os, sys
import torch.nn.functional as F
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

from wave_recurrent_unit import WaveRecurrentUnit, TOTAL_WAVE_DIM
from train_wru import load_phase1_and_phase2, load_phase2_5_checkpoint, PHASE2_5_CONFIG

log.cell_start("Cell 6 — Training Diagnostics (Phase 2.5)")

_checks   = {}
_has_fail = False

def _check(name, passed, warn=False, detail=''):
    global _has_fail
    if passed:
        print(f"  ✓ {name}")
    elif warn:
        print(f"  ⚠ {name}  {detail}")
    else:
        _has_fail = True
        print(f"  ✗ {name}  {detail}")
    _checks[name] = 'PASS' if passed else ('WARN' if warn else 'FAIL')

print("\n  ── Show-stopper checks ──────────────────")

# (1) Checkpoint exists
_ckpt_exists = os.path.exists(LOCAL_CKPT)
_check("Checkpoint exists on disk", _ckpt_exists, detail=LOCAL_CKPT)

# (2) Loads without error
_state = None
if _ckpt_exists:
    try:
        _state = torch.load(LOCAL_CKPT, map_location='cpu', weights_only=False)
        _check("Checkpoint loads without error", True)
    except Exception as e:
        _check("Checkpoint loads without error", False, detail=str(e))
else:
    _check("Checkpoint loads without error", False, detail="File missing")

# (3) state_dict keys present
if _state:
    _sd_keys = set(_state.get('state_dict', {}).keys())
    _has_wru = 'wru' in _sd_keys
    _check("state_dict contains 'wru' key", _has_wru, detail=f"keys: {_sd_keys}")
else:
    _check("state_dict contains 'wru' key", False, detail="Checkpoint not loaded")

# (4) Metrics present and finite
if _state:
    _metrics = _state.get('metrics', {})
    _gate_avg = _metrics.get('gate_avg', float('nan'))
    _gate_min = _metrics.get('gate_min', float('nan'))
    _check("Gate metric present and finite",
           math.isfinite(_gate_avg) and math.isfinite(_gate_min),
           detail=f"avg={_gate_avg} min={_gate_min}")
else:
    _gate_avg = float('nan')
    _gate_min = float('nan')
    _check("Gate metric present and finite", False)

# (5) WRU loads and forward passes
_wru_diag = None
try:
    _wru_diag = WaveRecurrentUnit()
    _wru_diag.load_state_dict(_state['state_dict']['wru'])
    _wru_diag.eval()
    _test_ctx = torch.randn(2, 512)
    _test_pred, _ = _wru_diag(_test_ctx)
    _check("WRU forward pass OK", _test_pred.shape == (2, TOTAL_WAVE_DIM),
           detail=f"shape={list(_test_pred.shape)}")
except Exception as e:
    _check("WRU forward pass OK", False, detail=str(e))

# (6) Output is not degenerate
if _wru_diag:
    _test_ctx2 = torch.randn(8, 512)
    _test_pred2, _ = _wru_diag(_test_ctx2)
    _std = _test_pred2.std().item()
    _check("Output not degenerate (std > 0.01)", _std > 0.01,
           warn=(_std <= 0.01), detail=f"std={_std:.6f}")
else:
    _check("Output not degenerate (std > 0.01)", False)

# (7) Energy bounded
if _wru_diag:
    _energy = (_test_pred2 ** 2).sum(dim=-1).max().item()
    _cap = PHASE2_5_CONFIG['energy_cap']
    _check(f"Energy bounded (< {_cap})", _energy <= _cap * 1.1,
           detail=f"max_energy={_energy:.1f}")
else:
    _check("Energy bounded", False)

# ── Save diagnostics ──────────────────────────────────────────────────────────
_diag_path = f'{RESULTS_DIR}/diagnostics.json'
with open(_diag_path, 'w') as _f:
    json.dump({
        'phase': 2.5,
        'version': 'v2',
        'checks': _checks,
        'gate_avg': _gate_avg if math.isfinite(_gate_avg) else None,
        'gate_min': _gate_min if math.isfinite(_gate_min) else None,
    }, _f, indent=2)
print(f"\n  ✓ Diagnostics saved → {_diag_path}")

if _has_fail:
    log.error("DIAGNOSTICS FAILED — fix issues before uploading")
    log.cell_end("Cell 6 — Training Diagnostics (Phase 2.5)", "FAIL")
    raise RuntimeError("Diagnostics FAILED — do not proceed to Cell 7.")
else:
    log.success("ALL DIAGNOSTICS PASSED — safe to upload")
    log.cell_end("Cell 6 — Training Diagnostics (Phase 2.5)", "PASS")

## Cell 7 — UPLOAD TO HUGGINGFACE
Uploads `checkpoints/phase2_5_v2.phase.pt` → `UnseenGAP/FLUX` at path `v2/phase2_5_v2.phase.pt`.

In [ ]:
# ── Cell 7: UPLOAD TO HUGGINGFACE ─────────────────────────────────────────────
import os
from huggingface_hub import HfApi

log.cell_start("Cell 7 — Upload to HuggingFace (Phase 2.5)")

# ── Re-resolve HF_TOKEN ───────────────────────────────────────────────────────
_hf_token = HF_TOKEN if 'HF_TOKEN' in dir() else ''
_hf_token = _hf_token or os.environ.get('HF_TOKEN', '')

if not _hf_token:
    try:
        from google.colab import userdata
        _hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        pass

if not _hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = (UserSecretsClient().get_secret('HF_TOKEN') or '').strip()
    except Exception:
        pass

if not _hf_token:
    log.error("HF_TOKEN is empty — add it via Colab/Kaggle secrets and re-run Cell 3")
    raise ValueError("HF_TOKEN is required for upload. Run Cell 3 first.")

HF_TOKEN = _hf_token
os.environ['HF_TOKEN'] = _hf_token
log.success(f"HF_TOKEN resolved (len={len(_hf_token)})")

# ── Upload ─────────────────────────────────────────────────────────────────────
_api = HfApi(token=_hf_token)

try:
    _api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    log.info(f"Repo confirmed: https://huggingface.co/{HF_REPO_ID}")
except Exception as _e:
    log.warning(f"create_repo: {_e}")

if not os.path.exists(LOCAL_CKPT):
    log.error(f"Checkpoint not found at {LOCAL_CKPT} — run Cell 5 first")
    raise FileNotFoundError(LOCAL_CKPT)

_ckpt_size_mb = os.path.getsize(LOCAL_CKPT) / 1e6
log.info(f"Uploading {_ckpt_size_mb:.1f} MB → {HF_REPO_ID}/{HF_CKPT_PATH}")

_upload_info = _api.upload_file(
    path_or_fileobj = LOCAL_CKPT,
    path_in_repo    = HF_CKPT_PATH,
    repo_id         = HF_REPO_ID,
    repo_type       = 'model',
    commit_message  = 'Phase 2.5 v2 — WRU (Wave Recurrent Unit) next-wave prediction',
)

_hf_url = f"https://huggingface.co/{HF_REPO_ID}/blob/main/{HF_CKPT_PATH}"
log.success("Upload complete")
log.metric("HF URL", _hf_url)
print(f"\n  HF checkpoint: {HF_CKPT_PATH}")
print(f"  Full URL: {_hf_url}")

log.cell_end("Cell 7 — Upload to HuggingFace (Phase 2.5)", "PASS")

## Cell 8 — TEST 1: Predicted Waves Decode to Readable Text
Loads checkpoint from HuggingFace. Tests that WRU output, when decoded by WTT,
produces valid bytes matching the ground-truth next chunk.
**Threshold:** avg byte accuracy ≥ 60%, min ≥ 30%.

In [ ]:
# ── Cell 8: TEST 1 — Decode Gate ──────────────────────────────────────────────
import sys, os, json, torch
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

from huggingface_hub import hf_hub_download
from train_wru import (
    load_phase1_and_phase2, load_phase2_5_checkpoint,
    run_phase2_5_decode_gate, PHASE2_5_CONFIG,
)
from flux_utils import PhaseResults

log.cell_start("Cell 8 — Test 1: Decode Gate (Phase 2.5)")

# ── Download checkpoints from HuggingFace ─────────────────────────────────────
for _hf_path, _name in [(HF_P1_PATH, "Phase 1"), (HF_P2_PATH, "Phase 2"), (HF_CKPT_PATH, "Phase 2.5")]:
    print(f"  Downloading {_name} from HuggingFace...")
    _dl = hf_hub_download(
        repo_id=HF_REPO_ID, filename=_hf_path,
        token=HF_TOKEN, local_dir=CHECKPOINT_DIR,
    )
    print(f"  ✓ {_name} → {_dl}")

# ── Load models ───────────────────────────────────────────────────────────────
_components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
_wru = load_phase2_5_checkpoint(device='cpu', hf_token=HF_TOKEN)
_wru.eval()

# ── Run decode gate ───────────────────────────────────────────────────────────
_passed, _avg, _min = run_phase2_5_decode_gate(
    cse=_components['cse'],
    chunker=_components['chunker'],
    w2f=_components['w2f'],
    wru=_wru,
    wtt=_components['wtt'],
    phase=2.5,
    verbose=True,
)

# ── Save metrics ──────────────────────────────────────────────────────────────
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)
_metrics['test1_gate_avg'] = round(_avg, 4)
_metrics['test1_gate_min'] = round(_min, 4)
_metrics['test1_passed']   = _passed
with open(_metrics_path, 'w') as _f:
    json.dump(_metrics, _f, indent=2)

# ── PhaseResults ──────────────────────────────────────────────────────────────
_results = PhaseResults(phase=2, component_name="WRU v2 (Phase 2.5)")
_results.add_test("Test1: Decode gate avg ≥ 60%", passed=_avg >= 0.60, score=_avg, threshold=0.60)
_results.add_test("Test1: Decode gate min ≥ 30%", passed=_min >= 0.30, score=_min, threshold=0.30)
_results.save()

if _passed:
    log.success(f"TEST 1 PASSED — avg={_avg:.1%}  min={_min:.1%}")
    log.cell_end("Cell 8 — Test 1: Decode Gate (Phase 2.5)", "PASS")
else:
    log.error(f"TEST 1 FAILED — avg={_avg:.1%}  min={_min:.1%}")
    log.cell_end("Cell 8 — Test 1: Decode Gate (Phase 2.5)", "FAIL")

## Cell 9 — TEST 2: Context Diversity (No Collapse)
Different prefix contexts must produce different predicted waves.
**Threshold:** avg pairwise cosine < 0.90 across 6 diverse texts.

In [ ]:
# ── Cell 9: TEST 2 — Context Diversity ────────────────────────────────────────
import sys, os, json, torch
import torch.nn.functional as F
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

from train_wru import load_phase1_and_phase2, load_phase2_5_checkpoint
from flux_utils import PhaseResults

log.cell_start("Cell 9 — Test 2: Context Diversity (Phase 2.5)")

DIVERSITY_TEXTS = [
    "The cat sat on the mat",
    "Quantum mechanics describes",
    "def fibonacci(n):",
    "Water freezes at zero degrees",
    "The stock market crashed",
    "Photosynthesis converts sunlight",
]

_components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
_wru = load_phase2_5_checkpoint(device='cpu', hf_token=HF_TOKEN)
_wru.eval()

_predictions = []
with torch.no_grad():
    for _txt in DIVERSITY_TEXTS:
        _wave = _components['cse'].encode(_txt)
        _cw, _ = _components['chunker'](_wave.full)
        if _cw.shape[0] < 2:
            continue
        _n = max(1, _cw.shape[0] // 2)
        _ctx = _components['w2f'](_cw[:_n].mean(dim=0)).unsqueeze(0)
        _pred, _ = _wru(_ctx)
        _predictions.append((_txt, _pred.squeeze(0)))

# Pairwise cosines
_cosines = []
print("\n  Pairwise cosine similarities:")
for _i in range(len(_predictions)):
    for _j in range(_i + 1, len(_predictions)):
        _cos = F.cosine_similarity(
            _predictions[_i][1].unsqueeze(0),
            _predictions[_j][1].unsqueeze(0),
            dim=-1,
        ).item()
        _cosines.append(_cos)
        _t1 = _predictions[_i][0][:25]
        _t2 = _predictions[_j][0][:25]
        _s = '✓' if _cos < 0.90 else '✗'
        print(f"  {_s} cos({_t1}, {_t2}) = {_cos:.4f}")

_avg_cos = sum(_cosines) / max(len(_cosines), 1)
_max_cos = max(_cosines) if _cosines else 1.0
print(f"\n  Avg pairwise cosine: {_avg_cos:.4f}  (want < 0.90)")
print(f"  Max pairwise cosine: {_max_cos:.4f}")

_diversity_pass = _avg_cos < 0.90

# Save metrics
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)
_metrics['test2_avg_cosine'] = round(_avg_cos, 4)
_metrics['test2_max_cosine'] = round(_max_cos, 4)
_metrics['test2_diversity_pass'] = _diversity_pass
with open(_metrics_path, 'w') as _f:
    json.dump(_metrics, _f, indent=2)

_results = PhaseResults(phase=2, component_name="WRU v2 (Phase 2.5)")
_results.add_test("Test2: avg pairwise cosine < 0.90", passed=_diversity_pass,
                  score=_avg_cos, threshold=0.90)
_results.save()

if _diversity_pass:
    log.success(f"TEST 2 PASSED — avg_cos={_avg_cos:.4f}")
    log.cell_end("Cell 9 — Test 2", "PASS")
else:
    log.error(f"TEST 2 FAILED — avg_cos={_avg_cos:.4f} ≥ 0.90 (context collapse)")
    log.cell_end("Cell 9 — Test 2", "FAIL")

## Cell 10 — TEST 3: WRU Output is a Valid Wave
Output must have correct shape [B, 432], bounded energy, and non-degenerate sub-bands.

In [ ]:
# ── Cell 10: TEST 3 — Wave Validity ───────────────────────────────────────────
import sys, os, json, torch
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

from wave_recurrent_unit import BAND_SLICES, TOTAL_WAVE_DIM
from train_wru import load_phase1_and_phase2, load_phase2_5_checkpoint, PHASE2_5_CONFIG
from flux_utils import PhaseResults

log.cell_start("Cell 10 — Test 3: Wave Validity (Phase 2.5)")

_components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
_wru = load_phase2_5_checkpoint(device='cpu', hf_token=HF_TOKEN)
_wru.eval()

TEST_TEXTS = [
    "The cat sat on the mat",
    "Quantum computing uses qubits",
    "def factorial(n):",
    "café résumé naïve",
    "Water is a polar molecule",
]

_all_ok = True
_energy_cap = PHASE2_5_CONFIG['energy_cap']

with torch.no_grad():
    for _txt in TEST_TEXTS:
        _wave = _components['cse'].encode(_txt)
        _cw, _ = _components['chunker'](_wave.full)
        if _cw.shape[0] < 2:
            print(f"  ⚠ Skipping '{_txt[:30]}' — too few chunks")
            continue
        _n = max(1, _cw.shape[0] // 2)
        _ctx = _components['w2f'](_cw[:_n].mean(dim=0)).unsqueeze(0)
        _pred, _ = _wru(_ctx)

        _energy = (_pred ** 2).sum().item()
        _max_abs = _pred.abs().max().item()

        _band_stds = {}
        _bands_ok = True
        for _name, (_s, _e) in BAND_SLICES.items():
            _std = _pred[0, _s:_e].std().item()
            _band_stds[_name] = _std
            if _std < 0.001:
                _bands_ok = False

        _ok = (_energy <= _energy_cap * 1.1) and (_max_abs < 5.0) and _bands_ok
        _status = '✓' if _ok else '✗'
        print(f"  {_status} '{_txt[:30]}'  energy={_energy:.2f}  max_abs={_max_abs:.3f}  "
              f"bands={', '.join(f'{n}:{s:.3f}' for n, s in _band_stds.items())}")
        if not _ok:
            _all_ok = False

    # Batch test
    _batch_ctx = torch.randn(16, 512)
    _batch_pred, _batch_state = _wru(_batch_ctx)
    assert _batch_pred.shape == (16, TOTAL_WAVE_DIM)
    _batch_max_e = (_batch_pred ** 2).sum(dim=-1).max().item()
    print(f"\n  ✓ Batch: shape={list(_batch_pred.shape)}  max_energy={_batch_max_e:.1f}")

# Save metrics
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)
_metrics['test3_wave_valid'] = _all_ok
_metrics['test3_batch_ok']   = True
with open(_metrics_path, 'w') as _f:
    json.dump(_metrics, _f, indent=2)

_results = PhaseResults(phase=2, component_name="WRU v2 (Phase 2.5)")
_results.add_test("Test3: All waves valid (energy, bands, shape)", passed=_all_ok,
                  score=1.0 if _all_ok else 0.0, threshold=1.0)
_results.save()

if _all_ok:
    log.success("TEST 3 PASSED — all wave validity checks OK")
    log.cell_end("Cell 10 — Test 3", "PASS")
else:
    log.error("TEST 3 FAILED — wave validity issues detected")
    log.cell_end("Cell 10 — Test 3", "FAIL")

## Cell 11 — DEMO 1: Prompt → WRU Prediction → Decoded Text
Shows the full pipeline for several texts, predicting the next word at each prefix length.

In [ ]:
# ── Cell 11: DEMO 1 — Next-Wave Prediction → Text ────────────────────────────
import sys, os, torch
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

from train_wru import load_phase1_and_phase2, load_phase2_5_checkpoint

log.cell_start("Cell 11 — Demo 1: Next-Wave Prediction (Phase 2.5)")

_components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
_wru = load_phase2_5_checkpoint(device='cpu', hf_token=HF_TOKEN)
_wru.eval()

DEMO_TEXTS = [
    "The cat sat on the mat",
    "Energy equals mass times the speed of light squared",
    "Machine learning models translate patterns into predictions",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1)",
]

with torch.no_grad():
    for _text in DEMO_TEXTS:
        _byte_data = _text.encode('utf-8')
        _wave = _components['cse'].encode(_text)
        _pairs = _components['chunker'].chunk_with_bytes(_wave.full, _byte_data)

        if len(_pairs) < 2:
            print(f"  ⚠ Skipping '{_text[:40]}' — too few chunks\n")
            continue

        _cw = torch.stack([p[0] for p in _pairs])
        _cb = [p[1] for p in _pairs]
        _ct = [b.decode('utf-8', errors='replace') for b in _cb]

        print(f"  ┌─ \"{_text}\"")
        print(f"  │  Chunks: {_ct}")

        for _n in range(1, len(_pairs)):
            _prefix_text = ''.join(_ct[:_n])
            _gt = _ct[_n] if _n < len(_ct) else "?"

            _ctx = _components['w2f'](_cw[:_n].mean(dim=0)).unsqueeze(0)
            _pred, _ = _wru(_ctx)
            _decoded = _components['wtt'].decode(_pred.squeeze(0), temperature=0.5)
            _decoded_text = _decoded.decode('utf-8', errors='replace')

            _match = '✓' if _decoded_text.strip() == _gt.strip() else '·'
            print(f"  │  [{_n}/{len(_pairs)}] \"{_prefix_text}\" → pred=\"{_decoded_text}\"  gt=\"{_gt}\"  {_match}")
        print(f"  └─\n")

log.cell_end("Cell 11 — Demo 1 (Phase 2.5)", "PASS")

## Cell 12 — DEMO 2: WRU Sub-Band Analysis
Shows per-band interference gates, energy, and how different contexts activate different bands.

In [ ]:
# ── Cell 12: DEMO 2 — Sub-Band Interference Analysis ─────────────────────────
import sys, os, torch
import torch.nn.functional as F
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

from wave_recurrent_unit import BAND_SLICES
from train_wru import load_phase1_and_phase2, load_phase2_5_checkpoint

log.cell_start("Cell 12 — Demo 2: Sub-Band Analysis (Phase 2.5)")

_components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
_wru = load_phase2_5_checkpoint(device='cpu', hf_token=HF_TOKEN)
_wru.eval()

DEMO_TEXTS = [
    "The cat sat on the mat",
    "def fibonacci(n): return n",
    "café résumé naïve",
]

# Parameter summary
_summary = _wru.parameter_summary()
print("\n  WRU Architecture:")
for _comp, _count in _summary.items():
    print(f"    {_comp:>20s}: {_count:>8,} params")

with torch.no_grad():
    for _text in DEMO_TEXTS:
        _byte_data = _text.encode('utf-8')
        _wave = _components['cse'].encode(_text)
        _pairs = _components['chunker'].chunk_with_bytes(_wave.full, _byte_data)

        if len(_pairs) < 2:
            continue

        _cw = torch.stack([p[0] for p in _pairs])
        _cb = [p[1] for p in _pairs]
        _n = max(1, len(_pairs) // 2)
        _ctx = _components['w2f'](_cw[:_n].mean(dim=0)).unsqueeze(0)
        _pred, _ = _wru(_ctx)

        _decoded = _components['wtt'].decode(_pred.squeeze(0), temperature=0.5)
        _decoded_text = _decoded.decode('utf-8', errors='replace')
        _gt = _cb[_n].decode('utf-8', errors='replace')

        print(f"\n  ┌─ \"{_text}\"")
        print(f"  │  pred=\"{_decoded_text}\"  gt=\"{_gt}\"")

        _ctx_wave = _wru.context_proj(_ctx).squeeze(0)
        _initial = _wru.get_initial_state(1).squeeze(0)

        print(f"  │  {'Band':<12s} {'Gate':>8s}  {'Initial σ':>9s}  {'Ctx σ':>8s}  {'Pred σ':>8s}")
        print(f"  │  {'─'*52}")

        for _name, (_s, _e) in BAND_SLICES.items():
            _sb = _initial[_s:_e]
            _cb2 = _ctx_wave[_s:_e]
            _pb = _pred.squeeze(0)[_s:_e]

            _gate = F.cosine_similarity(_sb.unsqueeze(0), _cb2.unsqueeze(0), dim=-1).item()
            _kind = "↑ constructive" if _gate > 0.1 else "↓ destructive" if _gate < -0.1 else "─ neutral"
            print(f"  │  {_name:<12s} {_gate:>+8.4f}  {_sb.std():.4f}    {_cb2.std():.4f}   {_pb.std():.4f}   {_kind}")

        _energy = (_pred ** 2).sum().item()
        print(f"  │  Energy: {_energy:.2f}  (cap: {_wru.energy_cap})")
        print(f"  └─")

log.cell_end("Cell 12 — Demo 2 (Phase 2.5)", "PASS")

## Cell 13 — SAVE RESULTS
Writes logs, metrics, and `RESULTS_PHASE_2_5.md` to `v2/V2_results/phase2_5/`.
Uploads logs to HuggingFace and pushes to GitHub.

In [ ]:
# ── Cell 13: SAVE RESULTS ─────────────────────────────────────────────────────
import os, sys, json, shutil, subprocess, re
sys.path.insert(0, REPO_PATH)

from flux_utils import PhaseResults, upload_logs_to_hf

log.cell_start("Cell 13 — Save Results (Phase 2.5)")

# ── Resolve credentials ────────────────────────────────────────────────────────
_GH_USER  = 'UnseenGAP'
_GH_TOKEN = GITHUB_TOKEN or os.environ.get('GITHUB_TOKEN', '')
if not _GH_TOKEN:
    try:
        from google.colab import userdata
        _GH_TOKEN = userdata.get('GITHUB_TOKEN') or ''
    except Exception:
        pass
if not _GH_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        _GH_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN') or ''
    except Exception:
        pass
_HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')
if not _GH_TOKEN:
    log.warning("GITHUB_TOKEN not found — git push will be skipped")

# ── 1. Copy log file ──────────────────────────────────────────────────────────
_log_src = f'{REPO_PATH}/logs/phase2.log'
_log_dst = f'{LOGS_DIR}/phase2_5.log'
if os.path.exists(_log_src):
    shutil.copy2(_log_src, _log_dst)
    log.success(f"Log copied → {_log_dst}")

# ── 2. Generate RESULTS_PHASE_2_5.md ──────────────────────────────────────────
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)

_results_final = PhaseResults(
    phase=2,
    component_name="Wave Recurrent Unit v2 (Phase 2.5 — Next-Wave Prediction)"
)
_results_final.add_test(
    "Test1: Decode gate avg ≥ 60%",
    passed=(_metrics.get('test1_gate_avg', 0) >= 0.60),
    score=_metrics.get('test1_gate_avg', 0), threshold=0.60,
)
_results_final.add_test(
    "Test1: Decode gate min ≥ 30%",
    passed=(_metrics.get('test1_gate_min', 0) >= 0.30),
    score=_metrics.get('test1_gate_min', 0), threshold=0.30,
)
_results_final.add_test(
    "Test2: Context diversity (avg cos < 0.90)",
    passed=(_metrics.get('test2_avg_cosine', 1.0) < 0.90),
    score=_metrics.get('test2_avg_cosine', 1.0), threshold=0.90,
)
_results_final.add_test(
    "Test3: Wave validity (energy, bands, shape)",
    passed=_metrics.get('test3_wave_valid', False),
    score=1.0 if _metrics.get('test3_wave_valid', False) else 0.0, threshold=1.0,
)
_results_final.save()
log.success("RESULTS_PHASE_2_5.md generated")

# ── 3. Scrub tokens + upload ──────────────────────────────────────────────────
_TOKEN_PATTERN = re.compile(
    r'\b(hf_[A-Za-z0-9]{10,}|ghp?_[A-Za-z0-9]{10,}|gh[oprs]_[A-Za-z0-9]{10,})\b'
)
def _scrub(path):
    if not os.path.exists(path): return 0
    with open(path, 'r', errors='replace') as f: content = f.read()
    scrubbed, count = _TOKEN_PATTERN.subn('[TOKEN_REDACTED]', content)
    if count:
        with open(path, 'w') as f: f.write(scrubbed)
    return count

for _sp in [_log_dst, _log_src]:
    _n = _scrub(_sp)
    if _n: print(f"  ✓ Scrubbed {_n} token(s) from {os.path.basename(_sp)}")

if _HF_TOKEN:
    try:
        upload_logs_to_hf(phase=2, hf_token=_HF_TOKEN)
        log.success("Logs uploaded to HuggingFace")
    except Exception as _e:
        log.warning(f"Log upload to HF failed: {_e}")

# ── 4. Git commit + push ──────────────────────────────────────────────────────
if not _GH_TOKEN:
    log.warning("Skipping git push — no GITHUB_TOKEN")
else:
    _remote_clean = None
    try:
        subprocess.run(['git', '-C', REPO_PATH, 'config', 'user.email', 'flux@unseengap.ai'], capture_output=True, timeout=10)
        subprocess.run(['git', '-C', REPO_PATH, 'config', 'user.name',  'FLUX Bot'],          capture_output=True, timeout=10)
        _remote_raw = subprocess.run(
            ['git', '-C', REPO_PATH, 'remote', 'get-url', 'origin'],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        _remote_clean = re.sub(r'https://[^@]+@', 'https://', _remote_raw)
        _auth_url = _remote_clean.replace('https://', f'https://{_GH_USER}:{_GH_TOKEN}@')
        subprocess.run(['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin', _auth_url], capture_output=True, timeout=10)
        subprocess.run(['git', '-C', REPO_PATH, 'fetch', 'origin', BRANCH],                  capture_output=True, timeout=30)
        subprocess.run(['git', '-C', REPO_PATH, 'reset', '--soft', f'origin/{BRANCH}'],       capture_output=True, timeout=10)

        for _f in ['logs/phase2.log', 'v2/V2_results/phase2_5']:
            subprocess.run(['git', '-C', REPO_PATH, 'add', '-f', _f], capture_output=True, timeout=10)

        _status = subprocess.run(
            ['git', '-C', REPO_PATH, 'status', '--porcelain'],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()

        if not _status:
            log.info("Nothing to commit")
        else:
            _commit = subprocess.run(
                ['git', '-C', REPO_PATH, 'commit', '-m',
                 'Phase 2.5 v2 — WRU results, logs [auto]'],
                capture_output=True, text=True, timeout=15,
            )
            if _commit.returncode == 0:
                log.success(f"git commit: {_commit.stdout.strip()[:80]}")

            _push = subprocess.run(
                ['git', '-C', REPO_PATH, 'push', 'origin', BRANCH],
                capture_output=True, text=True, timeout=60,
            )
            if _push.returncode == 0:
                log.success(f"Git push → github.com/{_GH_USER}/FLUX [{BRANCH}]")
            else:
                log.error(f"Git push FAILED:\n{_push.stderr.strip()[:300]}")

    except Exception as _git_err:
        log.error(f"Git error: {_git_err}")
    finally:
        if _remote_clean:
            subprocess.run(['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin', _remote_clean],
                           capture_output=True, timeout=10)

# ── 5. Directory tree ──────────────────────────────────────────────────────────
print(f"\n  v2/V2_results/phase2_5/")
for _root, _dirs, _files in os.walk(RESULTS_DIR):
    _level = _root.replace(RESULTS_DIR, '').count(os.sep)
    _indent = '  ' + '│   ' * _level + '├── '
    _rel = os.path.relpath(_root, RESULTS_DIR)
    if _rel != '.':
        print(f"{_indent}{os.path.basename(_root)}/")
    _sub = '  ' + '│   ' * (_level + 1) + '├── '
    for _fname in _files:
        _fpath = os.path.join(_root, _fname)
        _fsize = os.path.getsize(_fpath)
        print(f"{_sub}{_fname}  ({_fsize:,} bytes)")

log.cell_end("Cell 13 — Save Results (Phase 2.5)", "PASS")

## Cell 14 — FINAL SUMMARY

---

## FLUX v2 — Phase 2.5: Wave Recurrent Unit (WRU)

### What's Novel

The **Wave Recurrent Unit (WRU)** is a FLUX-native alternative to GRU/LSTM that uses
physics-inspired operations instead of learned gates:

| Feature | GRU | WRU |
|---------|-----|-----|
| Hidden state | Opaque vector | **Valid 432-dim wave** (decodable at every step) |
| Gates | sigmoid(Wx + Uh) | **Cosine interference** (physics-native) |
| Gating scope | Global (all dims) | **Per sub-band** (phonetic, syntactic, semantic, temporal, intensity) |
| Stability | Gradient clipping | **Energy conservation** (thermodynamic bound) |
| Interpretability | Black box | Each band independently inspectable |

### Sub-Bands

| Band | Dims | Role |
|------|------|------|
| phonetic | [0:64] | Sound/character patterns |
| syntactic | [64:128] | Grammar structure |
| semantic | [128:384] | Core meaning (largest band) |
| temporal | [384:416] | Position signals |
| intensity | [416:432] | Emphasis |

### Architecture

```
field_context [512]
  → context_proj → ctx_wave [432]
  → per-band cosine interference with wave_state
  → 5 band transforms (small MLPs)
  → energy-constrained superposition
  → output wave [432] (directly decodable by WTT)
```

### Test Results

| Test | Metric | Threshold | Status |
|------|--------|-----------|--------|
| Decode gate avg | See Cell 8 | ≥ 60% | See Cell 8 |
| Decode gate min | See Cell 8 | ≥ 30% | See Cell 8 |
| Context diversity | See Cell 9 | avg cos < 0.90 | See Cell 9 |
| Wave validity | See Cell 10 | energy bounded, bands non-degenerate | See Cell 10 |

### Artifacts

| Artifact | Path |
|----------|------|
| Checkpoint (local) | `checkpoints/phase2_5_v2.phase.pt` |
| Checkpoint (HF) | [`UnseenGAP/FLUX → v2/phase2_5_v2.phase.pt`](https://huggingface.co/UnseenGAP/FLUX/blob/main/v2/phase2_5_v2.phase.pt) |
| Phase 1 (frozen) | `v2/phase1_v2.phase.pt` |
| Phase 2 (frozen) | `v2/phase2_v2.phase.pt` |
| Logs | `v2/V2_results/phase2_5/logs/` |
| Metrics | `v2/V2_results/phase2_5/metrics.json` |
| Results | `v2/V2_results/phase2_5/RESULTS_PHASE_2_5.md` |

### What Was Proved

- A FLUX-native recurrent unit can predict the **next wave** from prefix context
- **No GRU/LSTM needed** — cosine interference + energy conservation are sufficient
- Output is ALWAYS a valid SemanticWave — decodable at any step
- Per-band gating allows independent control of syntax vs semantics vs phonetics
- Different contexts produce genuinely different predictions (no collapse)

### Next Phase

**Phase 3 v2 — Autoregressive Wave Generation** (`phase3_v2.ipynb`)
Chain WRU predictions: each output feeds back as context for the next step.
Add REINFORCE for train/inference alignment.

> HuggingFace: [UnseenGAP/FLUX](https://huggingface.co/UnseenGAP/FLUX)
> GitHub: [Unseengap/FLUX @ v2](https://github.com/Unseengap/FLUX/tree/v2)